# Lets look deeper into r and the LD score. 

We'll write our our own r and LD score not based on tskit. Then we will investigate why these scores have certain properties.
- why the vertical line when conpared to clade correlation ?
- why the solid line coming up ?
- what kinda of snps create these patterns ?

In [ ]:
import tskit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, SVG

In [ ]:
def calculate_r(a, b):
    if len(a) != len(b) or len(a) == 0:
        raise ValueError("Lists must have the same nonzero length.")

    mean_a = sum(a) / len(a)
    mean_b = sum(b) / len(b)

    numerator = sum((x - mean_a) * (y - mean_b)
                    for x, y in zip(a, b))

    denominator = (
        sum((x - mean_a)**2 for x in a)
        * sum((y - mean_b)**2 for y in b)
    ) ** 0.5

    if denominator == 0:
        raise ValueError("r is undefined when either site has no variation.")

    return numerator / denominator

In [ ]:
def ld_score(focal, snps):
    return sum(calculate_r(focal, snp)**2 for snp in snps)

In [ ]:
def plot_ld_vs_max_corr(results, metric="r2", ax=None):
    columns = {
        "r2": "max_r2",
        "ld_score": "ld_score",
    }

    labels = {
        "r2": r"Maximum $r^2$ with retained SNPs",
        "ld_score": r"LD score (sum of $r^2$)",
    }

    if metric not in labels:
        raise ValueError("metric must be 'r2' or 'ld_score'")

    column = columns[metric]
    data = results[[column, "max_corr"]].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4))

    ax.scatter(
        data[column],
        data["max_corr"],
        s=12,
        edgecolors="none",
        rasterized=True,
    )

    ax.set_xlabel(labels[metric])
    ax.set_ylabel("Maximum correlation (max_corr)")
    ax.set_title(f"{len(data):,} SNPs")
    ax.spines[["top", "right"]].set_visible(False)

    return ax

In [ ]:
a = [0, 0, 0, 0, 1, 1, 1, 1]
b = [0, 0, 0, 1, 0, 1, 1, 1]
c = [0, 1, 0, 1, 0, 1, 0, 1]

r = calculate_r(a, b)
print(r)      # 0.5
print(r**2)   # 0.25


score = ld_score(a, [a, b, c])
print(score) 

In [ ]:
# Load the tree sequences

filename = "chr12_HapMapII_GRCh37_CEU500_seed1"
percent = "10"

# reference ts from stdpopsim
ts_std = tskit.load(f"../stdpopsim_simulations/simulations/{filename}.trees")
# masked ts from Relate
ts_masked = tskit.load(f"../Relate_sims/masked_{filename}_{percent}.00percent_25bins_seed42.trees")

# load in hidden SNP indices
hidden_snps_df = pd.read_csv(f"../Relate_sims/hidden_snps_key_{filename}_{percent}.00percent_25bins_seed42.csv")

# load in the full .hapsfile
path = f"../Relate_sims/data/{filename}.haps"

# format the haps file into a pandas dataframe
haps_df = pd.read_csv(f"../Relate_sims/data/{filename}.haps", sep=r"\s+", header=None)
# format the haps file into a pandas dataframe
masked_haps_df = pd.read_csv(f"../Relate_sims/data/masked_{filename}_{percent}.00percent_25bins_seed42.haps", sep=r"\s+", header=None)

haps_df = haps_df.rename(columns={
    0: "Chromosome number",
    1: "SNP ID",
    2: "SNP position",
    3: "Ancestral allele",
    4: "Alternative allele",
})

masked_haps_df = masked_haps_df.rename(columns={
    0: "Chromosome number",
    1: "SNP ID",
    2: "SNP position",
    3: "Ancestral allele",
    4: "Alternative allele",
})


In [ ]:
# laod results filename previously calculated
results_filename = "chr12_HapMapII_GRCh37_CEU500_seed1_10.00percent500000bp_25bins_seed42"

genetic_map = pd.read_csv(
    "../Relate_sims/genetic_maps/genetic_map_GRCh37_chr12.txt", sep=r"\s+"
).sort_values("Position(bp)")

results = pd.read_csv(f"{results_filename}_results.csv")
results

In [ ]:
haps_df

In [ ]:
masked_haps_df

In [ ]:
hidden_snps_df

In [ ]:
# laod results filename previously calculated
results_filename = "chr12_HapMapII_GRCh37_CEU500_seed1_10.00percent500000bp_25bins_seed42"

genetic_map = pd.read_csv(
    "../Relate_sims/genetic_maps/genetic_map_GRCh37_chr12.txt", sep=r"\s+"
).sort_values("Position(bp)")

results = pd.read_csv(f"{results_filename}_results.csv")
results

In [ ]:
if "optimized_max_r2" not in results.columns:    
    # window we are looking at for each snp
    window_bp = 500_000
    # what are the positions of the SNPs that are not masked in the masked haps file. these are the only eligible ones
    unmasked_positions = set(masked_haps_df["SNP position"])

    # Tree-sequence sites are ordered by position.
    eligible_sites = [
        site for site in ts_std.sites()
        if site.position in unmasked_positions
    ]
    eligible_positions = np.array([site.position for site in eligible_sites])
    eligible_ids = np.array([site.id for site in eligible_sites], dtype=int)

    #initialize lists to store the results
    n_retained_values = []
    max_r2_values = []
    ld_score_values = []
    top10_r2_sum_values = []

    for position in results["SNP position"]:
        focal_site = ts_std.site(position=position)

        left = np.searchsorted(
            eligible_positions, position - window_bp, side="left"
        )
        right = np.searchsorted(
            eligible_positions, position + window_bp, side="right"
        )

        # what are the eligible (ones that arent masked) SNPs in the window around the focal SNP, excluding the focal SNP itself
        retained_ids = eligible_ids[left:right]
        retained_ids = retained_ids[retained_ids != focal_site.id]

        n_retained_values.append(len(retained_ids))

        if len(retained_ids) == 0:
            max_r2_values.append(np.nan)
            ld_score_values.append(np.nan)
            top10_r2_sum_values.append(np.nan)
            continue

        ld_values = ts_std.ld_matrix(
            sites=[[focal_site.id], retained_ids.tolist()],
            mode="site",
            stat="r2",
        )

        r2 = np.asarray(ld_values).reshape(-1)
        valid_r2 = r2[np.isfinite(r2)]

        if valid_r2.size == 0:
            max_r2_values.append(np.nan)
            ld_score_values.append(np.nan)
            top10_r2_sum_values.append(np.nan)
        else:
            max_r2_values.append(valid_r2.max())
            ld_score_values.append(valid_r2.sum())

            k = min(10, valid_r2.size)
            top_k = np.partition(valid_r2, valid_r2.size - k)[-k:]
            top10_r2_sum_values.append(top_k.sum())

    results = results.copy()
    results["optimized_n_retained_snps"] = n_retained_values
    results["optimized_max_r2"] = max_r2_values
    results["optimized_ld_score"] = ld_score_values
    results["optimized_top10_r2_sum"] = top10_r2_sum_values

    results.to_csv(f"{results_filename}_results.csv", index=False)

In [ ]:
results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(results["max_corr"], results["optimized_max_r2"], s=4, alpha=0.2)
ax.set_xlabel("max_corr")
ax.set_ylabel("max_r2")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(results["max_corr"], results["optimized_ld_score"], s=4, alpha=0.2)
ax.set_xlabel("max_corr")
ax.set_ylabel("optimized_ld_score")
fig.tight_layout()
plt.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(7, 6))
# hb = ax.hexbin(results["max_corr"], results["max_r2"], gridsize=60, bins="log", mincnt=1, alpha=0.8, cmap="viridis")
# fig.colorbar(hb, ax=ax, label="Number of SNPs")
# ax.set_xlabel("max_corr")
# ax.set_ylabel("max_r2")
# fig.tight_layout()
# plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(results["max_corr"], results["optimized_top10_r2_sum"], s=4, alpha=0.2)
ax.set_xlabel("max_corr")
ax.set_ylabel("optimized_top10_r2_sum")
fig.tight_layout()
plt.show()

### lets inspect the weird stuff


In [ ]:
## first the max r w the low max corr
# what kind of snos make this ? how many are singletons ? how many are doubletons ? how many are in the middle ? what the number of retained snps is for these ? 
# what is the ld score for these ? 


In [ ]:
results

In [ ]:
# make the not close version
tol = 1e-3

results["not_close"] = (
    results["max_corr"].notna()
    & ~np.isclose(results["max_corr"], 1, atol=tol, rtol=0)
)

not_close = results.loc[results["not_close"]]

display(not_close)

In [ ]:
df_max_r2_1_all = results.loc[results["max_r2"] == 1].copy()
df_max_r2_1_not_close = not_close.loc[not_close["max_r2"] == 1].copy()
print(f"What percent of results are in df_max_r2_1_all? {df_max_r2_1_all.shape[0]/results.shape[0]:.2%}")
print(f"What percent of results are in df_max_r2_1_not_close? {df_max_r2_1_not_close.shape[0]/not_close.shape[0]:.2%}")

In [ ]:
alt_counts_all = df_max_r2_1_all["ALT_bin_num"].value_counts().sort_index()
alt_counts_not_close = df_max_r2_1_not_close["ALT_bin_num"].value_counts().sort_index()


fig, axes = plt.subplots(1, 2, figsize=(15, 4), sharey=False)

for ax, counts, title in zip(
    axes,
    [alt_counts_all, alt_counts_not_close],
    ["All SNPs (max_r2 = 1)", "Not-close SNPs (max_r2 = 1)"],
):
    ax.bar(counts.index, counts.values)
    ax.set_xlabel("ALT_bin_num")
    ax.set_title(title)

axes[0].set_ylabel("Number of SNPs")
fig.tight_layout()
plt.show()

In [ ]:
df_max_r2_1_not_close

In [ ]:
# are the n_retained_snps  for these SNPs different from the rest of the  SNPs?
# no theyre not
print(df_max_r2_1_not_close["n_retained_snps"].agg(["min", "median", "mean"]))
print(results["n_retained_snps"].agg(["min", "median", "mean"]))

In [ ]:
df = df_max_r2_1_not_close
bins = sorted(df["ALT_bin_num"].unique())
data = [df.loc[df["ALT_bin_num"] == b, "max_corr"] for b in bins]

fig, ax = plt.subplots(figsize=(12, 4))
ax.boxplot(data, positions=range(len(bins)), showfliers=False)

# Overlay the individual SNPs, since some bins will have few points.
rng = np.random.default_rng(0)
for i, d in enumerate(data):
    ax.scatter(i + rng.uniform(-0.2, 0.2, len(d)), d, s=4, alpha=0.3, color="grey")

ax.set_xticks(range(len(bins)))
ax.set_xticklabels(bins)
ax.set_xlabel("ALT_bin_num")
ax.set_ylabel("max_corr")
ax.set_title("Distribution of max_corr for SNPs with max_r2 = 1 (not-close) by ALT_bin_num")
fig.tight_layout()
plt.show()

In [ ]:
df = df_max_r2_1_not_close
bins = sorted(df["ALT_bin_num"].unique())
data = [df.loc[df["ALT_bin_num"] == b, "ld_score"] for b in bins]

fig, ax = plt.subplots(figsize=(12, 4))
ax.boxplot(data, positions=range(len(bins)), showfliers=False)

rng = np.random.default_rng(0)
for i, d in enumerate(data):
    ax.scatter(i + rng.uniform(-0.2, 0.2, len(d)), d, s=4, alpha=0.3, color="grey")

ax.set_xticks(range(len(bins)))
ax.set_xticklabels(bins)
ax.set_xlabel("ALT_bin_num")
ax.set_ylabel("LD score")
ax.set_yscale("log")  # LD scores are right-skewed; remove for a linear axis
ax.set_title("Distribution of LD scores for SNPs with max_r2 = 1 (not-close) by ALT_bin_num")
fig.tight_layout()
plt.show()

In [ ]:
high = df_max_r2_1_not_close
low = results[results["max_r2"] < 0.7]

bins = sorted(set(high["ALT_bin_num"]) | set(low["ALT_bin_num"]))
x = np.arange(len(bins))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 4))
for df, offset, color, label in [
    (high, -w / 2, "tab:blue", "max_r2 = 1"),
    (low, w / 2, "tab:orange", "max_r2 < 0.7"),
]:
    data = [df.loc[df["ALT_bin_num"] == b, "ld_score"] for b in bins]
    ax.boxplot(
        data,
        positions=x + offset,
        widths=w * 0.9,
        showfliers=False,
        patch_artist=True,
        boxprops=dict(facecolor=color, alpha=0.5),
        medianprops=dict(color="black"),
    )
    ax.plot([], [], color=color, alpha=0.5, linewidth=8, label=label)

ax.set_xticks(x)
ax.set_xticklabels(bins)
ax.set_xlabel("ALT_bin_num")
ax.set_ylabel("LD score")
ax.set_yscale("log")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
bins = np.linspace(results["SNP position"].min(), results["SNP position"].max(), 100)

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(results["SNP position"] / 1e6, bins=bins / 1e6, alpha=0.4, label="All SNPs")
ax.hist(df_max_r2_1_not_close["SNP position"] / 1e6, bins=bins / 1e6, alpha=0.8, label="max_r2 = 1, not close")
ax.set_xlabel("SNP position (Mb)")
ax.set_ylabel("Number of SNPs")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
background = results
highlight = df_max_r2_1_not_close

fig, ax = plt.subplots(figsize=(12, 4))

ax.scatter(
    background["SNP position"], background["max_corr"],
    s=8, color="lightgrey", alpha=0.4, rasterized=True,
)

sc = ax.scatter(
    highlight["SNP position"], highlight["max_corr"],
    c=highlight["ALT_bin_num"], cmap="viridis",
    vmin=results["ALT_bin_num"].min(), vmax=results["ALT_bin_num"].max(),
    s=14, alpha=0.8,
)

fig.colorbar(sc, ax=ax, label="Allele-count bin")
ax.set_xlabel("SNP position")
ax.set_ylabel("Maximum clade–carrier correlation (φ)")
ax.set_ylim(0, 1.05)
ax.set_title("SNPs with max_r2 = 1 (not-close) highlighted by ALT_bin_num")
fig.tight_layout()
plt.show()

In [ ]:
# whats up w the 10 r^2 ???

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(results["max_corr"], results["optimized_top10_r2_sum"], s=4, alpha=0.2)
ax.set_xlabel("max_corr")
ax.set_ylabel("optimized_top10_r2_sum")
fig.tight_layout()
plt.show()

In [ ]:
df_top10_low_corr = results[
    np.isclose(results["optimized_top10_r2_sum"], 10)
    & (results["max_corr"] < 0.6)
].copy()

df_top10_low_corr

In [ ]:
counts = df_top10_low_corr["ALT_bin_num"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(counts.index, counts.values)
ax.set_xlabel("ALT_bin_num")
ax.set_ylabel("Number of SNPs")
ax.yaxis.set_major_locator(plt.MultipleLocator(5))
ax.set_title("SNPs with top-10 r² sum = 10 and max_corr < 0.6")
fig.tight_layout()
plt.show()

In [ ]:
df = df_top10_low_corr
bins = sorted(df["ALT_bin_num"].unique())
data = [df.loc[df["ALT_bin_num"] == b, "max_corr"] for b in bins]

fig, ax = plt.subplots(figsize=(12, 4))
ax.boxplot(data, positions=range(len(bins)), showfliers=False)

# Overlay the individual SNPs, since some bins will have few points.
rng = np.random.default_rng(0)
for i, d in enumerate(data):
    ax.scatter(i + rng.uniform(-0.2, 0.2, len(d)), d, s=4, alpha=0.3, color="grey")

ax.set_xticks(range(len(bins)))
ax.set_xticklabels(bins)
ax.set_xlabel("ALT_bin_num")
ax.set_ylabel("max_corr")
ax.set_title("Distribution of max_corr for SNPs with max_r2 = 1 (not-close) by ALT_bin_num")
fig.tight_layout()
plt.show()

In [ ]:
background = results
highlight = df_top10_low_corr

fig, ax = plt.subplots(figsize=(12, 4))

ax.scatter(
    background["SNP position"], background["max_corr"],
    s=8, color="lightgrey", alpha=0.4, rasterized=True,
)

sc = ax.scatter(
    highlight["SNP position"], highlight["max_corr"],
    c=highlight["ALT_bin_num"], cmap="viridis",
    vmin=results["ALT_bin_num"].min(), vmax=results["ALT_bin_num"].max(),
    s=14, alpha=0.8,
)

fig.colorbar(sc, ax=ax, label="Allele-count bin")
ax.set_xlabel("SNP position")
ax.set_ylabel("Maximum clade–carrier correlation (φ)")
ax.set_ylim(0, 1.05)
ax.set_title("SNPs with max_r2 = 1 (not-close) highlighted by ALT_bin_num")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
hb = ax.hexbin(
    results["optimized_ld_score"], results["optimized_top10_r2_sum"],
    xscale="log", gridsize=60, bins="log", mincnt=1,
)
fig.colorbar(hb, ax=ax, label="Number of SNPs")
ax.set_xlabel("LD score")
ax.set_ylabel("Top-10 r² sum")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(
    not_close["optimized_ld_score"], not_close["optimized_top10_r2_sum"],
    c=not_close["ALT_bin_num"], cmap="viridis",
    s=5, alpha=0.4, rasterized=True,
)
fig.colorbar(sc, ax=ax, label="Allele-count bin")
ax.set_xscale("log")
ax.set_xlabel("LD score")
ax.set_ylabel("Top-10 r² sum")
fig.tight_layout()
plt.show()

In [ ]:
df = not_close.sort_values("max_corr", ascending=False)  # Draw low max_corr last, so it's on top.

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(
    df["optimized_ld_score"], df["optimized_top10_r2_sum"],
    c=df["max_corr"], cmap="viridis", vmin=0, vmax=1,
    s=5, alpha=0.4, rasterized=True,
)
fig.colorbar(sc, ax=ax, label="max_corr")
ax.set_xscale("log")
ax.set_xlabel("LD score")
ax.set_ylabel("Top-10 r² sum")
fig.tight_layout()
plt.show()

In [ ]:
df = results.sort_values("max_corr", ascending=False)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(projection="3d")
sc = ax.scatter(
    np.log10(df["optimized_ld_score"]),
    df["optimized_top10_r2_sum"],
    df["ALT_bin_num"],
    c=df["max_corr"], cmap="viridis", vmin=0, vmax=1,
    s=4, alpha=0.4,
)
fig.colorbar(sc, ax=ax, label="max_corr", shrink=0.6, pad=0.1)
ax.set_xlabel("log10(LD score)")
ax.set_ylabel("Top-10 r² sum")
ax.set_zlabel("ALT_bin_num")
ax.view_init(elev=20, azim=-60)  # Change these to rotate the view.
plt.show()

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    not_close.assign(log_ld=np.log10(not_close["optimized_ld_score"])),
    x="log_ld", y="optimized_top10_r2_sum", z="ALT_bin_num",
    color="max_corr", opacity=0.4,
)
fig.update_traces(marker_size=2)
fig.show()